# Bangladesh: 0235_totram

Static adaptation only; Earth Engine/data/asset execution not verified.

Original scientific assumptions and date windows retained; use the national workflow for QA improvements.

Review scale, missing data, units, and any training/calibration requirements before interpreting results.

Experimental model: use spatial/time holdout validation and independent Bangladesh ground truth; finer pixels do not prove added information.

May contain tutorial-specific assets or imported training features; inspect dependencies before execution.

In [ ]:
# Bangladesh adaptation: 0235_totram
# Original author: Amirhossein Ahrari; see original attribution below and originals/.
from bdgee.core import initialize, boundaries
from bdgee.notebook import export_outputs, upload_local
initialize()  # Run `earthengine authenticate` first; project is configured in bdgee.
BD_COUNTRY, BD_DISTRICTS = boundaries()
BD_ROI = BD_COUNTRY.geometry()
BD_BBOX = [88.0, 20.5, 92.7, 26.7]  # search only, not analysis boundary
roi = geometry = border = iran = tehran = province = loc = point = BD_ROI
# Run from the repository root; all local outputs are isolated by source.
from pathlib import Path
import os
_REPO_ROOT = Path.cwd()
_OUTPUT_DIR = _REPO_ROOT / 'outputs' / 'soil_drought' / '0235_totram'
_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(_OUTPUT_DIR)


In [ ]:
"""00024_landsat_totram.ipynb

Tutorial Code by Amirhossein Ahrari
YouTube: https://www.youtube.com/@amirhosseinahrarigee
Tutorial Video: Google Earth Engine Tutorial-211: Landsat Soil Moisture Estimation using NDVI and LST (TOTRAM)
This code is part of a tutorial series on Earth Engine programming techniques
presented on the Amirhossein Ahrari YouTube channel. You are free to use and modify
this code for academic and non-academic purposes. Don't forget to subscribe to
the Amirhossein Ahrari channel and follow the videos to support the instructor

"""
import ee
None
None
'# **Study Area**'
import geemap
map = geemap.Map(basemap='SATELLITE')
map
roi = BD_ROI
roi
'# **Landsat9 Image Collection**'
landsat9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterDate('2025', '2026').filterBounds(roi).filter(ee.Filter.lt('CLOUD_COVER', 45))
landsat9

def ndvi_lst(img):
    qa_band = img.select('QA_PIXEL')
    bit1 = qa_band.bitwiseAnd(1 << 1).neq(0)
    bit2 = qa_band.bitwiseAnd(1 << 2).neq(0)
    bit3 = qa_band.bitwiseAnd(1 << 3).neq(0)
    bit4 = qa_band.bitwiseAnd(1 << 4).neq(0)
    mask = bit1.Or(bit2).Or(bit3).Or(bit4)
    sr = img.select('SR_B*.').multiply(2.75e-05).add(-0.2)
    ndvi = sr.normalizedDifference(['SR_B5', 'SR_B4']).rename('ndvi')
    lst = img.select('ST_B10').multiply(0.00341802).add(149).rename('lst')
    stack = ee.Image.cat([ndvi, lst])
    return stack.updateMask(mask.Not()).copyProperties(img, ['system:time_start'])
landsat_ndvi_lst = landsat9.map(ndvi_lst)
landsat_ndvi_lst
landsat9.first().select('SR_B4').projection()
'# **to Xarray**'
from xee import helpers
from shapely.geometry import shape
import xarray as xr
grid_30m = helpers.fit_geometry(geometry=shape(roi.getInfo()), grid_crs='EPSG:32638', grid_scale=(30, -30))
ds = xr.open_dataset(landsat_ndvi_lst, engine='ee', **grid_30m)
ds = ds.sortby('time') * 1
ds
'# **Temporal Resampling**'
ds_monthly = ds.resample(time='M').max('time')
ds_smooth = ds_monthly.rolling(time=3, center=True, min_periods=1).mean('time')
ds_mean = ds_smooth.mean('time')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ds_mean['ndvi'].plot(x='x', y='y', robust=True, ax=ax[0])
ds_mean['lst'].plot(x='x', y='y', robust=True, ax=ax[1])
plt.tight_layout()
'# **TOTRAM**'
from sklearn.linear_model import QuantileRegressor
import numpy as np
x = ds_mean.ndvi.values.flatten()
x = x[~np.isnan(x)]
y = ds_mean.lst.values.flatten()
y = y[~np.isnan(y)]
x = x.reshape(-1, 1)
samples = np.random.choice(len(x), 20000, replace=False)
x_sample = x[samples]
y_sample = y[samples]
dry_edge = QuantileRegressor(quantile=0.95, alpha=0).fit(x_sample, y_sample)
wet_edge = QuantileRegressor(quantile=0.05, alpha=0).fit(x_sample, y_sample)
s_d = dry_edge.coef_[0]
i_d = dry_edge.intercept_
s_w = wet_edge.coef_[0]
i_w = wet_edge.intercept_
lst_d = i_d + s_d * ds_mean.ndvi
lst_w = i_w + s_w * ds_mean.ndvi
sm1 = (lst_d - ds_mean.lst) / (lst_d - lst_w)
sm1 = sm1.clip(0, 1)
sm1.plot()
sm2 = (i_d + s_d * ds_mean.ndvi - ds_mean.lst) / (i_d - i_w + (s_d - s_w) * ds_mean.ndvi)
sm2 = sm2.clip(0, 1)
sm2.plot()


In [ ]:

# Review these task plans; task creation does not start exports.
os.chdir(_REPO_ROOT)
bd_tasks = export_outputs(globals(), 'soil_drought', '0235_totram', scale=1000)
# Set an appropriate native scale above before creating plans.
# for task in bd_tasks: task.start()
# upload_local(_OUTPUT_DIR, 'soil_drought')  # explicit upload; requires Drive OAuth scope
